In [30]:
import sys
if 'google.colab' in sys.modules:
    !pip --quiet install open-atmos-jupyter-utils
    from open_atmos_jupyter_utils import pip_install_on_colab
    pip_install_on_colab('PyMPDATA-examples')

In [31]:
import numpy as np
from matplotlib import pyplot
from open_atmos_jupyter_utils import show_plot, show_anim
from PyMPDATA import ScalarField, Solver, Stepper, VectorField, Options, boundary_conditions

In [32]:
class ShallowWaterEquationsIntegrator:
    def __init__(self, *, h_initial: np.ndarray, options: Options = None, bathymetry: np.ndarray, vh_initial: np.ndarray, uh_initial: np.ndarray):
        """ initializes the solvers for a given initial condition of `h` assuming zero momenta at t=0 """
        options = options or Options(nonoscillatory=True, infinite_gauge=True)

        X, Y, grid = 0, 1, h_initial.shape
        stepper = Stepper(options=options, grid=grid, n_threads=1)
        kwargs = {
            'boundary_conditions': [boundary_conditions.Constant(value=0), boundary_conditions.Constant(value = 0)],
            'halo': options.n_halo,
        }
        advectees = {
            "h": ScalarField(h_initial, **kwargs),
            "uh": ScalarField(uh_initial, **kwargs),
            "vh": ScalarField(vh_initial, **kwargs),
        }
        self.advector = VectorField((
                np.zeros((grid[X] + 1, grid[Y])),
                np.zeros((grid[X], grid[Y] + 1))
            ), **kwargs
        )
        self.solvers = { k: Solver(stepper, v, self.advector) for k, v in advectees.items() }
        self.bathymetry = bathymetry
        self.max_courant_numbers = []
    def __getitem__(self, key):
        """ returns `key` advectee field of the current solver state """
        return self.solvers[key].advectee.get()

    def _apply_half_rhs(self, *, key, axis, g_times_dt_over_dxy):
        """ applies half of the source term in the given direction """
        self[key][:] -= .5 * g_times_dt_over_dxy * self['h'] * np.gradient(self['h']-self.bathymetry, axis=axis)

    def _update_courant_numbers(self, *, axis, key, mask, dt_over_dxy):
        """ computes the Courant number component from fluid column height and momenta fields """
        velocity = np.where(mask, np.nan, 0)
        momentum = self[key]
        np.divide(momentum, self['h'], where=mask, out=velocity)

        all = slice(None, None)
        all_but_last = slice(None, -1)
        all_but_first_and_last = slice(1, -1)

        velocity_at_cell_boundaries = velocity[(
            (all_but_last, all),
            (all, all_but_last),
        )[axis]] + np.diff(velocity, axis=axis) / 2
        courant_number = self.advector.get_component(axis)[(
            (all_but_first_and_last, all),
            (all, all_but_first_and_last)
        )[axis]]
        courant_number[:] = velocity_at_cell_boundaries * dt_over_dxy[axis]
        assert np.amax(np.abs(courant_number)) <= 1
        return np.amax(np.abs(courant_number))
    def __call__(self, *, nt: int, g: float, dt_over_dxy: tuple, outfreq: int, eps: float=1e-7):
        """ integrates `nt` timesteps and returns a dictionary of solver states recorded every `outfreq` step[s] """
        output = {k: [] for k in self.solvers.keys()}
        for it in range(nt + 1):
            if it != 0:
                mask = self['h'] > eps
                max_c_axis = []
                for axis, key in enumerate(("uh", "vh")):
                    max_c_axis.append(self._update_courant_numbers(axis=axis, key=key, mask=mask, dt_over_dxy=dt_over_dxy))
                self.max_courant_numbers.append(max(max_c_axis))
                self.solvers["h"].advance(n_steps=1)
                for axis, key in enumerate(("uh", "vh")):
                    self._apply_half_rhs(key=key, axis=axis, g_times_dt_over_dxy=g * dt_over_dxy[axis])
                    self.solvers[key].advance(n_steps=1)
                    self._apply_half_rhs(key=key, axis=axis, g_times_dt_over_dxy=g * dt_over_dxy[axis])
            if it % outfreq == 0:
                for key in self.solvers.keys():
                    output[key].append(self[key].copy())
        return output


In [ ]:
grid = (150, 40)
v_speed = 0
bathymetry = np.tile(np.linspace(0,3, grid[1]), (grid[0], 1))
h_initial = bathymetry.copy()
h_initial[
grid[0]//3:grid[0]-grid[0]//3,
grid[1]-grid[1]//20:grid[1]
] += .025

h_min_depth = 1e-3
h_initial = np.maximum(h_initial, h_min_depth)

vh_initial = np.zeros(grid)
vh_initial[
grid[0]//3:grid[0]-grid[0]//3,
grid[1]-grid[1]//20:grid[1]
] += v_speed

uh_initial = np.zeros(grid)
integrator = ShallowWaterEquationsIntegrator(
    h_initial=h_initial, bathymetry=bathymetry, vh_initial=vh_initial, uh_initial=uh_initial
)
output = integrator(
    nt= 1200, g=10, dt_over_dxy=(.0125, .0125), outfreq=30, eps=h_min_depth
)


In [ ]:
def plot(frame, *, zlim=(-.25, .25)):
    psi = output['h'][frame]-bathymetry
    xi, yi = np.indices(psi.shape)
    fig, ax = pyplot.subplots(subplot_kw={"projection": "3d"}, figsize=(12, 6))
    ax.plot_wireframe(xi+.5, yi+.5, psi, color='blue', linewidth=.5)
    ax.set(zlim=zlim, proj_type='ortho', title=f"t / Δt = {frame}", zlabel=r"$\zeta$")
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.fill = False
        axis.pane.set_edgecolor('black')
        axis.pane.set_alpha(1)
    for axis in ('x', 'y'):
        getattr(ax, f'set_{axis}label')(f"{axis} / Δ{axis}")
    pyplot.colorbar(
        ax.contourf(xi+.5, yi+.5, bathymetry, zdir='z', offset=zlim[0]),
        pad=.1, aspect=10, fraction=.02, label='bathymetry', location='left'
    ).ax.invert_yaxis()
    return fig
show_anim(plot, range(len(output['h'])))

In [ ]:
num_y = grid[1]

max_psi_at_y = -np.inf * np.ones(num_y)

for frame_data in output['h']:
    psi_frame = frame_data - bathymetry

    psi_at_x_mid = psi_frame[grid[0]//2, :]

    max_psi_at_y = np.maximum(max_psi_at_y, psi_at_x_mid)

y_top_10_percent_start = num_y - (num_y // 10)
highest_psi_in_top_10_percent = np.max(max_psi_at_y[y_top_10_percent_start:])

if highest_psi_in_top_10_percent != 0:
    k = 1 / highest_psi_in_top_10_percent
else:
    k = 1.0

scaled_y_indices = np.arange(num_y) * k
scaled_max_psi_at_y = max_psi_at_y * k

max_depth_for_alpha = 3.0
alpha = np.degrees(np.arcsin(max_depth_for_alpha / grid[1]))

pyplot.figure(figsize=(12, 6))
pyplot.plot(scaled_y_indices, scaled_max_psi_at_y)
pyplot.title(f'Wysokość fali od odległości od brzegu dla alpha = {alpha:.2f}°')
pyplot.xlabel('Odległość od brzegu [m]')
pyplot.ylabel('Wysokość fali [m]')
pyplot.grid(True)
pyplot.show()


Testy stabilności

In [ ]:
grid = (150, 40)
v_speed = 0
bathymetry = np.tile(np.linspace(0,3, grid[1]), (grid[0], 1))
h_initial = bathymetry.copy()
h_initial[
grid[0]//3:grid[0]-grid[0]//3,
grid[1]-grid[1]//20:grid[1]
] += .025

h_min_depth = 1e-3
h_initial = np.maximum(h_initial, h_min_depth)

vh_initial = np.zeros(grid)
vh_initial[
grid[0]//3:grid[0]-grid[0]//3,
grid[1]-grid[1]//20:grid[1]
] += v_speed

uh_initial = np.zeros(grid)
integrator = ShallowWaterEquationsIntegrator(
    h_initial=h_initial, bathymetry=bathymetry, vh_initial=vh_initial, uh_initial=uh_initial
)
output = integrator(
    nt= 600, g=10, dt_over_dxy=(.025, .025), outfreq=1, eps=h_min_depth
)


In [ ]:
num_y = grid[1]

max_psi_at_y = -np.inf * np.ones(num_y)

for frame_data in output['h']:
    psi_frame = frame_data - bathymetry
    psi_at_x_mid = psi_frame[grid[0]//2, :]

    max_psi_at_y = np.maximum(max_psi_at_y, psi_at_x_mid)

y_top_10_percent_start = num_y - (num_y // 10)
highest_psi_in_top_10_percent = np.max(max_psi_at_y[y_top_10_percent_start:])

if highest_psi_in_top_10_percent != 0:
    k = 1 / highest_psi_in_top_10_percent
else:
    k = 1.0

scaled_y_indices = np.arange(num_y) * k
scaled_max_psi_at_y = max_psi_at_y * k

max_depth_for_alpha = 3.0
alpha = np.degrees(np.arcsin(max_depth_for_alpha / grid[1]))

pyplot.figure(figsize=(12, 6))
pyplot.plot(scaled_y_indices, scaled_max_psi_at_y)
pyplot.title(f'Wysokość fali od odległości od brzegu dla podwojonego dt/dxy i alpha = {alpha:.2f}°')
pyplot.xlabel('Odległość od brzegu [m]')
pyplot.ylabel('Wysokość fali [m]')
pyplot.grid(True)
pyplot.show()


In [ ]:
grid = (300, 80)
v_speed = 0
bathymetry = np.tile(np.linspace(0,3, grid[1]), (grid[0], 1))
h_initial = bathymetry.copy()
h_initial[
grid[0]//3:grid[0]-grid[0]//3,
grid[1]-grid[1]//20:grid[1]
] += .025

h_min_depth = 1e-3
h_initial = np.maximum(h_initial, h_min_depth)

vh_initial = np.zeros(grid)
vh_initial[
grid[0]//3:grid[0]-grid[0]//3,
grid[1]-grid[1]//20:grid[1]
] += v_speed

uh_initial = np.zeros(grid)
integrator = ShallowWaterEquationsIntegrator(
    h_initial=h_initial, bathymetry=bathymetry, vh_initial=vh_initial, uh_initial=uh_initial
)
output = integrator(
    nt= 2400, g=10, dt_over_dxy=(.0125, .0125), outfreq=1, eps=h_min_depth
)


In [ ]:
num_y = grid[1]

max_psi_at_y = -np.inf * np.ones(num_y)

for frame_data in output['h']:
    psi_frame = frame_data - bathymetry
    psi_at_x_mid = psi_frame[grid[0]//2, :]

    max_psi_at_y = np.maximum(max_psi_at_y, psi_at_x_mid)

y_top_10_percent_start = num_y - (num_y // 10)
highest_psi_in_top_10_percent = np.max(max_psi_at_y[y_top_10_percent_start:])

if highest_psi_in_top_10_percent != 0:
    k = 1 / highest_psi_in_top_10_percent
else:
    k = 1.0

scaled_y_indices = np.arange(num_y) * k
scaled_max_psi_at_y = max_psi_at_y * k

max_depth_for_alpha = 3.0
alpha = np.degrees(np.arcsin(max_depth_for_alpha / grid[1]))

pyplot.figure(figsize=(12, 6))
pyplot.plot(scaled_y_indices, scaled_max_psi_at_y)
pyplot.title(f'Wysokość fali od odległości od brzegu dla podwojonej rozdzielczości przestrzennej i alpha = {alpha:.2f}°')
pyplot.xlabel('Odległość od brzegu [m]')
pyplot.ylabel('Wysokość fali [m]')
pyplot.grid(True)
pyplot.show()


Test dla najmniejszego skosu

In [ ]:
grid = (150, 40)
v_speed = 0
bathymetry = np.tile(np.linspace(0,0.1, grid[1]), (grid[0], 1))
h_initial = bathymetry.copy()
h_initial[
grid[0]//3:grid[0]-grid[0]//3,
grid[1]-grid[1]//20:grid[1]
] += .025

h_min_depth = 1e-3
h_initial = np.maximum(h_initial, h_min_depth)

vh_initial = np.zeros(grid)
vh_initial[
grid[0]//3:grid[0]-grid[0]//3,
grid[1]-grid[1]//20:grid[1]
] += v_speed

uh_initial = np.zeros(grid)
integrator = ShallowWaterEquationsIntegrator(
    h_initial=h_initial, bathymetry=bathymetry, vh_initial=vh_initial, uh_initial=uh_initial
)
output = integrator(
    nt=4500, g=10, dt_over_dxy=(.0125, .0125), outfreq=1, eps=h_min_depth
)


In [ ]:
num_y = grid[1]

max_psi_at_y = -np.inf * np.ones(num_y)

for frame_data in output['h']:
    psi_frame = frame_data - bathymetry
    psi_at_x_mid = psi_frame[grid[0]//2, :]

    max_psi_at_y = np.maximum(max_psi_at_y, psi_at_x_mid)

y_top_10_percent_start = num_y - (num_y // 10)
highest_psi_in_top_10_percent = np.max(max_psi_at_y[y_top_10_percent_start:])

if highest_psi_in_top_10_percent != 0:
    k = 1 / highest_psi_in_top_10_percent
else:
    k = 1.0

scaled_y_indices = np.arange(num_y) * k
scaled_max_psi_at_y = max_psi_at_y * k

max_depth_for_alpha = 0.5
alpha = np.degrees(np.arcsin(max_depth_for_alpha / grid[1]))

pyplot.figure(figsize=(12, 6))
pyplot.plot(scaled_y_indices, scaled_max_psi_at_y)
pyplot.title(f'Wysokość fali od odległości od brzegu dla alpha = {alpha:.2f}°')
pyplot.xlabel('Odległość od brzegu [m]')
pyplot.ylabel('Wysokość fali [m]')
pyplot.grid(True)
pyplot.show()


Końcowa symulacja

In [ ]:
max_bathymetry_depths = np.arange(0.1,3.05,0.1)

all_max_depths = []
all_wave_distances = []

for current_max_depth in max_bathymetry_depths:
    nt = int(np.interp(current_max_depth, [0.5, 3.0], [4500, 1200]))

    grid = (150, 40)
    v_speed = 0
    bathymetry = np.tile(np.linspace(0, current_max_depth, grid[1]), (grid[0], 1))
    h_initial = bathymetry.copy()
    h_initial[
        grid[0]//3:grid[0]-grid[0]//3,
        grid[1]-grid[1]//20:grid[1]
    ] += .025

    h_min_depth = 1e-3
    h_initial = np.maximum(h_initial, h_min_depth)

    vh_initial = np.zeros(grid)
    vh_initial[
        grid[0]//3:grid[0]-grid[0]//3,
        grid[1]-grid[1]//20:grid[1]
    ] += v_speed

    uh_initial = np.zeros(grid)
    integrator = ShallowWaterEquationsIntegrator(
        h_initial=h_initial, bathymetry=bathymetry, vh_initial=vh_initial, uh_initial=uh_initial
    )
    output = integrator(
        nt=nt, g=10, dt_over_dxy=(.0125, .0125), outfreq=1, eps=h_min_depth
    )

    num_y = grid[1]
    max_psi_at_y = -np.inf * np.ones(num_y)

    for frame_data in output['h']:
        psi_frame = frame_data - bathymetry
        psi_at_x_mid = psi_frame[grid[0]//2, :]
        max_psi_at_y = np.maximum(max_psi_at_y, psi_at_x_mid)

    y_top_10_percent_start = num_y - (num_y // 10)
    highest_psi_in_top_10_percent = np.max(max_psi_at_y[y_top_10_percent_start:])

    if highest_psi_in_top_10_percent != 0:
        k = 1 / highest_psi_in_top_10_percent
    else:
        k = 1.0

    scaled_y_indices = np.arange(num_y) * k
    scaled_max_psi_at_y = max_psi_at_y * k

    y_peak_idx_original = np.argmax(scaled_max_psi_at_y)

    threshold_scaled = 0.5
    y_end_idx = -1

    for i in range(y_peak_idx_original - 1, -1, -1):
        if scaled_max_psi_at_y[i] < threshold_scaled:
            y_end_idx = i
            break

    if scaled_max_psi_at_y[y_peak_idx_original] < threshold_scaled:
        wave_distance_scaled = 0.0
    elif y_end_idx != -1:
        wave_distance_scaled = scaled_y_indices[y_peak_idx_original] - scaled_y_indices[y_end_idx]
    else:
        wave_distance_scaled = scaled_y_indices[y_peak_idx_original] - scaled_y_indices[0]

    all_max_depths.append(current_max_depth)
    all_wave_distances.append(wave_distance_scaled)


In [ ]:
grid_y_dim_for_angle_calc = 40
angle_of_beach = np.arcsin(np.array(all_max_depths) / grid_y_dim_for_angle_calc)

pyplot.figure(figsize=(10, 6))
pyplot.plot(np.degrees(angle_of_beach), all_wave_distances, marker='o', linestyle='-')
pyplot.title('Długość surfowania vs. Kąt plaży liniowej')
pyplot.xlabel('Kąt plaży liniowej [°]')
pyplot.ylabel('Długość surfowania [m]')
pyplot.grid(True)
pyplot.show()
